In [36]:
ORIGINATION_COLUMNS = [
    "CREDIT_SCORE",                    # 1
    "FIRST_PAYMENT_DATE",              # 2  — post-décision
    "FIRST_TIME_HOMEBUYER_FLAG",       # 3
    "MATURITY_DATE",                   # 4  — post-décision
    "MSA",                             # 5
    "MI_PERCENTAGE",                   # 6
    "NUMBER_OF_UNITS",                 # 7
    "OCCUPANCY_STATUS",                # 8
    "OCLTV",                           # 9
    "DTI",                             # 10
    "ORIGINAL_UPB",                    # 11
    "LTV",                             # 12
    "ORIGINAL_INTEREST_RATE",          # 13
    "CHANNEL",                         # 14
    "PPM_FLAG",                        # 15
    "PRODUCT_TYPE",                    # 16
    "STATE",                           # 17
    "PROPERTY_TYPE",                   # 18
    "POSTAL_CODE",                     # 19
    "LOAN_SEQUENCE_NUMBER",            # 20 — post-décision (clé Freddie Mac)
    "LOAN_PURPOSE",                    # 21
    "ORIGINAL_LOAN_TERM",              # 22
    "NUMBER_OF_BORROWERS",             # 23
    "SELLER_NAME",                     # 24
    "SERVICER_NAME",                   # 25
    "SUPER_CONFORMING_FLAG",           # 26
    "PRE_RELIEF_REFI_LOAN_SEQ",        # 27
    "PROGRAM_INDICATOR",               # 28
    "RELIEF_REFINANCE_INDICATOR",      # 29
    "PROPERTY_VALUATION_METHOD",       # 30
    "IO_FLAG",                         # 31
    "MORTGAGE_INSURANCE_CANCELLATION", # 32
]



In [37]:
# Séparation post-décision / éligibles :
# python# Post-décision — exclure du scoring

ORIGINATION_POST_DECISION = [
    "FIRST_PAYMENT_DATE",       # connu après déblocage des fonds
    "MATURITY_DATE",            # déduit après octroi
    "LOAN_SEQUENCE_NUMBER",     # généré par Freddie Mac après rachat
]

In [38]:
# Éligibles — inputs à la décision
ORIGINATION_ELIGIBLE = [
    "CREDIT_SCORE",
    "FIRST_TIME_HOMEBUYER_FLAG",
    "MSA",
    "MI_PERCENTAGE",
    "NUMBER_OF_UNITS",
    "OCCUPANCY_STATUS",
    "OCLTV",
    "DTI",
    "ORIGINAL_UPB",
    "LTV",
    "ORIGINAL_INTEREST_RATE",
    "CHANNEL",
    "PPM_FLAG",
    "PRODUCT_TYPE",
    "STATE",
    "PROPERTY_TYPE",
    "POSTAL_CODE",
    "LOAN_PURPOSE",
    "ORIGINAL_LOAN_TERM",
    "NUMBER_OF_BORROWERS",
    "SELLER_NAME",
    "SERVICER_NAME",
    "SUPER_CONFORMING_FLAG",
    "PRE_RELIEF_REFI_LOAN_SEQ",
    "PROGRAM_INDICATOR",
    "RELIEF_REFINANCE_INDICATOR",
    "PROPERTY_VALUATION_METHOD",
    "IO_FLAG",
    "MORTGAGE_INSURANCE_CANCELLATION",
]

In [4]:
import pandas as pd
import numpy as np








files = {
    "2007Q1": "/Users/macbookpro/platform/Backend/data/historical_data_2007Q1/historical_data_2007Q1.txt",
    "2010Q1": "/Users/macbookpro/platform/Backend/data/historical_data_2010Q1/historical_data_2010Q1.txt",
    "2020Q1": "/Users/macbookpro/platform/Backend/data/historical_data_2020Q1/historical_data_2020Q1.txt",
    "2025Q1": "/Users/macbookpro/platform/Backend/data/historical_data_2025/historical_data_2025Q1/historical_data_2025Q1.txt",

}

dfs = []
for vintage, path in files.items():
    df = pd.read_csv(
        path,
        sep="|",
        header=None,
        names=ORIGINATION_COLUMNS,
        low_memory=False
    )
    df['VINTAGE'] = vintage
    dfs.append(df)
    print(f"{vintage} → {df.shape[0]:,} lignes / {df['LOAN_SEQUENCE_NUMBER'].nunique():,} prêts")

real = pd.concat(dfs, ignore_index=True)

2007Q1 → 302,331 lignes / 302,331 prêts
2010Q1 → 360,856 lignes / 360,856 prêts
2020Q1 → 517,072 lignes / 517,072 prêts
2025Q1 → 210,749 lignes / 210,749 prêts


# REMPLACER Les valeur de convention

Gestion des convention

In [39]:
real.CREDIT_SCORE = real.CREDIT_SCORE.replace(9999, np.nan)

In [6]:
real['FIRST_TIME_HOMEBUYER_FLAG'] = real['FIRST_TIME_HOMEBUYER_FLAG'].astype(object)
real['FIRST_TIME_HOMEBUYER_FLAG'] = real['FIRST_TIME_HOMEBUYER_FLAG'].replace(9, np.nan)


In [7]:
real['MI_PERCENTAGE'] = real['MI_PERCENTAGE'].replace(999, np.nan)

In [8]:
real['NUMBER_OF_UNITS'] = real['NUMBER_OF_UNITS'].replace(99, np.nan)

In [9]:
real['OCCUPANCY_STATUS'] = real['OCCUPANCY_STATUS'].replace(9, np.nan)
real['OCCUPANCY_STATUS'] = real['OCCUPANCY_STATUS'].astype(object)


In [10]:
real['OCLTV'] = real['OCLTV'].replace(999, np.nan)

In [11]:
real['DTI'] = real['DTI'].replace(999, np.nan)

In [12]:
real['LTV'] = real['LTV'].replace(999, np.nan)

In [13]:
real['CHANNEL'] = real['CHANNEL'].replace(9, np.nan)

In [14]:
real['PPM_FLAG'] = real['PPM_FLAG'].astype(object)
real['PPM_FLAG'].unique()


array(['N', 'Y'], dtype=object)

In [15]:
real['PRODUCT_TYPE'].unique()

array(['FRM'], dtype=object)

In [16]:
real['PROPERTY_TYPE'].unique()

array(['SF', 'PU', 'MH', 'CO', 'CP'], dtype=object)

In [17]:
real['PROPERTY_TYPE'] = real['PROPERTY_TYPE'].replace(99, np.nan)

In [18]:
real['LOAN_PURPOSE'] = real['LOAN_PURPOSE'].replace(9, np.nan)

In [19]:
real['NUMBER_OF_BORROWERS'] = real['NUMBER_OF_BORROWERS'].replace(99, np.nan)

In [20]:
real['PROGRAM_INDICATOR'] = real['PROGRAM_INDICATOR'].astype(object)

In [21]:
real['IO_FLAG'].unique()

array(['N'], dtype=object)

In [34]:
real['MSA'].unique

<bound method Series.unique of 0          20260.0
1          36980.0
2          48424.0
3          31860.0
4              NaN
            ...   
1391003    39540.0
1391004        NaN
1391005    41740.0
1391006    24340.0
1391007        NaN
Name: MSA, Length: 1391008, dtype: float64>

In [22]:
real['MORTGAGE_INSURANCE_CANCELLATION'] = real['MORTGAGE_INSURANCE_CANCELLATION'].astype(object)


In [44]:
real['ORIGINAL_LOAN_TERM']

array([240, 180, 360, 120, 300, 480, 144, 356, 159, 306, 354, 132, 168,
       294, 292, 359, 289, 290, 204, 158, 178, 358, 352, 288, 355, 291,
       366, 348, 239, 350, 357, 283, 307, 296, 157,  60, 353, 264, 324,
       312, 330, 260, 297, 364, 286, 276, 156, 299, 177, 351, 361, 119,
       333, 175,  84, 319, 183,  96, 287, 249, 214, 216, 228, 252, 343,
       342, 347, 241, 277, 313, 186, 336, 349, 251, 154, 337, 176, 270,
       420, 293, 346, 600, 174, 108, 316, 233, 284, 344, 460, 238, 236,
       635, 250, 232, 303, 123, 327, 234, 192, 171, 206, 406, 285, 328,
       317, 315, 269, 338, 335, 362, 220, 235, 179,  72, 170, 334, 237,
       340, 345, 169, 339, 219, 155, 329, 332, 149, 148, 331, 323, 320,
       322, 321, 301, 215, 254, 314, 318, 221, 309, 121, 136, 164, 282,
       280, 341, 162, 163, 161, 126, 325, 145, 130, 165, 304, 139, 298,
       302, 129, 210, 281, 137, 166, 135, 152, 173, 160, 217, 305, 172,
       271, 295, 310, 278, 190, 231, 230, 261, 140, 127, 153, 13

# Gestion des valeur manquante

In [23]:
real.isna().mean()

CREDIT_SCORE                       0.000361
FIRST_PAYMENT_DATE                 0.000000
FIRST_TIME_HOMEBUYER_FLAG          0.000000
MATURITY_DATE                      0.000000
MSA                                0.143279
MI_PERCENTAGE                      0.000003
NUMBER_OF_UNITS                    0.000000
OCCUPANCY_STATUS                   0.000000
OCLTV                              0.000035
DTI                                0.075064
ORIGINAL_UPB                       0.000000
LTV                                0.000027
ORIGINAL_INTEREST_RATE             0.000000
CHANNEL                            0.000000
PPM_FLAG                           0.000000
PRODUCT_TYPE                       0.000000
STATE                              0.000000
PROPERTY_TYPE                      0.000000
POSTAL_CODE                        0.000000
LOAN_SEQUENCE_NUMBER               0.000000
LOAN_PURPOSE                       0.000000
ORIGINAL_LOAN_TERM                 0.000000
NUMBER_OF_BORROWERS             

In [24]:
real.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1391008 entries, 0 to 1391007
Data columns (total 33 columns):
 #   Column                           Non-Null Count    Dtype  
---  ------                           --------------    -----  
 0   CREDIT_SCORE                     1390506 non-null  float64
 1   FIRST_PAYMENT_DATE               1391008 non-null  int64  
 2   FIRST_TIME_HOMEBUYER_FLAG        1391008 non-null  object 
 3   MATURITY_DATE                    1391008 non-null  int64  
 4   MSA                              1191706 non-null  float64
 5   MI_PERCENTAGE                    1391004 non-null  float64
 6   NUMBER_OF_UNITS                  1391008 non-null  int64  
 7   OCCUPANCY_STATUS                 1391008 non-null  object 
 8   OCLTV                            1390960 non-null  float64
 9   DTI                              1286594 non-null  float64
 10  ORIGINAL_UPB                     1391008 non-null  int64  
 11  LTV                              1390970 non-null 

In [25]:
real.describe()

,CREDIT_SCORE,FIRST_PAYMENT_DATE,MATURITY_DATE,MSA,MI_PERCENTAGE,NUMBER_OF_UNITS,OCLTV,DTI,ORIGINAL_UPB,LTV,ORIGINAL_INTEREST_RATE,POSTAL_CODE,ORIGINAL_LOAN_TERM,NUMBER_OF_BORROWERS,PROPERTY_VALUATION_METHOD
count,1.390506e+06,1.391008e+06,1.391008e+06,1.191706e+06,1.391004e+06,1.391008e+06,1.390960e+06,1.286594e+06,1.391008e+06,1.390970e+06,1.391008e+06,1.391008e+06,1.391008e+06,1.390856e+06,1.391008e+06
mean,7.480530e+02,2.015380e+05,2.042497e+05,3.024162e+04,4.721273e+00,1.027529e+00,7.251100e+01,3.507923e+01,2.462531e+05,7.106909e+01,5.041301e+00,5.401429e+04,3.255295e+02,1.517691e+00,4.270029e+00
std,4.871188e+01,6.687377e+02,9.048154e+02,1.127226e+04,1.021391e+01,2.163126e-01,1.872932e+01,1.060843e+01,1.456960e+05,1.811411e+01,1.269901e+00,2.978884e+04,6.891705e+01,5.125056e-01,2.624427e+00
min,3.000000e+02,2.007010e+05,2.012020e+05,1.018000e+04,0.000000e+00,1.000000e+00,3.000000e+00,1.000000e+00,8.000000e+03,3.000000e+00,2.000000e+00,0.000000e+00,6.000000e+01,1.000000e+00,1.000000e+00
25%,7.170000e+02,2.010030e+05,2.037030e+05,1.912400e+04,0.000000e+00,1.000000e+00,6.200000e+01,2.700000e+01,1.380000e+05,6.000000e+01,3.875000e+00,2.960000e+04,3.600000e+02,1.000000e+00,2.000000e+00
50%,7.590000e+02,2.020030e+05,2.040040e+05,3.154000e+04,0.000000e+00,1.000000e+00,7.600000e+01,3.600000e+01,2.130000e+05,7.500000e+01,5.000000e+00,5.310000e+04,3.600000e+02,2.000000e+00,2.000000e+00
75%,7.870000e+02,2.020050e+05,2.050040e+05,3.974000e+04,0.000000e+00,1.000000e+00,8.400000e+01,4.300000e+01,3.240000e+05,8.000000e+01,6.125000e+00,8.340000e+04,3.600000e+02,2.000000e+00,7.000000e+00
max,8.500000e+02,2.025100e+05,2.062060e+05,4.974000e+04,5.000000e+01,4.000000e+00,4.360000e+02,6.500000e+01,2.300000e+06,2.530000e+02,9.250000e+00,9.990000e+04,6.350000e+02,5.000000e+00,7.000000e+00


In [26]:
# LTV / OCLTV — cohérence


# Combien de cas ?
mask_incoherent = real['OCLTV'] < real['LTV']
print(f"Cas incohérents : {mask_incoherent.sum()}")

# Voir les cas
real[mask_incoherent][['LOAN_SEQUENCE_NUMBER', 'LTV', 'OCLTV', 'VINTAGE']].head(20)

Cas incohérents : 0


,LOAN_SEQUENCE_NUMBER,LTV,OCLTV,VINTAGE


In [27]:
# SUPER_CONFORMING_FLAG — vide = non super conforme
real['SUPER_CONFORMING_FLAG'] = real['SUPER_CONFORMING_FLAG'].fillna('N')

# RELIEF_REFINANCE_INDICATOR — vide = non applicable
real['RELIEF_REFINANCE_INDICATOR'] = real['RELIEF_REFINANCE_INDICATOR'].fillna('N')

# OCLTV — remplacer par LTV si manquant
real['OCLTV'] = real['OCLTV'].fillna(real['LTV'])

# LTV — remplacer par OCLTV si manquant
real['LTV'] = real['LTV'].fillna(real['OCLTV'])

# NUMBER_OF_BORROWERS — médiane = 1
real['NUMBER_OF_BORROWERS'] = real['NUMBER_OF_BORROWERS'].fillna(1)

# Vérification
print(real[['SUPER_CONFORMING_FLAG', 'RELIEF_REFINANCE_INDICATOR',
            'OCLTV', 'LTV', 'NUMBER_OF_BORROWERS']].isna().sum())

SUPER_CONFORMING_FLAG          0
RELIEF_REFINANCE_INDICATOR     0
OCLTV                         38
LTV                           38
NUMBER_OF_BORROWERS            0
dtype: int64


In [28]:
# Indicatrices avant imputation
real['IS_MISSING_CREDIT_SCORE'] = real['CREDIT_SCORE'].isna().astype(int)
real['IS_MISSING_DTI'] = real['DTI'].isna().astype(int)

# Imputation par médiane par segment (VINTAGE x PROPERTY_TYPE)
real['CREDIT_SCORE'] = real.groupby(['VINTAGE', 'PROPERTY_TYPE'])['CREDIT_SCORE']\
                           .transform(lambda x: x.fillna(x.median()))

real['DTI'] = real.groupby(['VINTAGE', 'PROPERTY_TYPE'])['DTI']\
                  .transform(lambda x: x.fillna(x.median()))

# Vérification
print(real[['CREDIT_SCORE', 'DTI',
            'IS_MISSING_CREDIT_SCORE', 'IS_MISSING_DTI']].isna().sum())

CREDIT_SCORE               0
DTI                        0
IS_MISSING_CREDIT_SCORE    0
IS_MISSING_DTI             0
dtype: int64


# Save data

In [29]:
ORIGINATION_ELIGIBLE = ORIGINATION_ELIGIBLE + ['IS_MISSING_CREDIT_SCORE', 'IS_MISSING_DTI', 'LOAN_SEQUENCE_NUMBER']

In [30]:
to_drop = [col for col in real.columns if  col not in ORIGINATION_ELIGIBLE]
to_drop

['FIRST_PAYMENT_DATE', 'MATURITY_DATE', 'VINTAGE']

In [31]:
real = real.drop(to_drop, axis=1)

In [43]:
real.columns

Index(['CREDIT_SCORE', 'FIRST_TIME_HOMEBUYER_FLAG', 'MSA', 'MI_PERCENTAGE',
       'NUMBER_OF_UNITS', 'OCCUPANCY_STATUS', 'OCLTV', 'DTI', 'ORIGINAL_UPB',
       'LTV', 'ORIGINAL_INTEREST_RATE', 'CHANNEL', 'PPM_FLAG', 'PRODUCT_TYPE',
       'STATE', 'PROPERTY_TYPE', 'POSTAL_CODE', 'LOAN_SEQUENCE_NUMBER',
       'LOAN_PURPOSE', 'ORIGINAL_LOAN_TERM', 'NUMBER_OF_BORROWERS',
       'SELLER_NAME', 'SERVICER_NAME', 'SUPER_CONFORMING_FLAG',
       'PRE_RELIEF_REFI_LOAN_SEQ', 'PROGRAM_INDICATOR',
       'RELIEF_REFINANCE_INDICATOR', 'PROPERTY_VALUATION_METHOD', 'IO_FLAG',
       'MORTGAGE_INSURANCE_CANCELLATION', 'IS_MISSING_CREDIT_SCORE',
       'IS_MISSING_DTI'],
      dtype='object')

In [33]:
real.to_csv("/Users/macbookpro/platform/Backend/data/processed/origination_v01.csv", index=False,sep=",")